In [1]:
import numpy as np
import matplotlib as plt
from typing import List, Tuple

In [ ]:
class MultiArmedBandit:
    """A Simple Multi-Armed Bandit Problem Implementation."""
    def __init__(self, arm_probabilities: List[float]):
        """
        Initialize the Multi-Armed Bandit with bernouli arms probabilities.

        :param arm_probabilities: List of probabilities for each arm.
        """
        

        
        self.arm_probabilities = arm_probabilities
        self.num_arms = len(arm_probabilities)
        self.optimal_arm = np.argmax(arm_probabilities)
    
    def pull(self, arm: int) -> int:
        """
        Pull the specified arm and return a reward(0 or 1).

        :param arm: Index of the arm to pull.
        :return: 1 if success, 0 otherwise.
        """
        if arm >= self.num_arms or arm < 0:
            raise ValueError(f"Arm index {arm} is out of bounds. Must be between 0 and {self.num_arms - 1}.")
        return int(np.random.random() < self.arm_probabilities[arm])
    def get_optimal_arm(self) -> int:
        """        Get the index of the optimal arm.
        :return: Index of the optimal arm.
        """
        return np.argmax(self.arm_probabilities)

bandit = MultiArmedBandit([0.95, 0.90, 0.10])
    
    
        

In [5]:
class GreedyAgent:
    """A Greedy Agent always exploits."""
    def __init__(self, num_arms: int):
        """
        Initialize the Greedy Agent.

        :param num_arms: Number of arms in the bandit problem.
        """
        self.num_arms = num_arms
        self.arm_counts = np.zeros(num_arms) # number of times each was pulled
        self.values = np.zeros(num_arms) # estiamted value of each arm
       
    def select_arm(self) -> int:
        
        """ select the arm with highest estimated value(with random tie breaking) """
        
        # for arms never pulled, assume value 0
        
        max_value = np.max(self.values)
        best_arms = np.where(self.values == max_value)[0]
        return np.random.choice(best_arms) if len(best_arms) > 1 else best_arms[0]
    
    def update(self, arm: int, reward: int):
        """
        Update the agent's knowledge based on the reward received from pulling an arm.

        :param arm: Index of the arm that was pulled.
        :param reward: Reward received from pulling the arm (0 or 1).
        """
        self.arm_counts[arm] += 1
        n = self.arm_counts[arm]
        # Update estimated value using incremental formula
        self.values[arm] += (reward - self.values[arm]) / n
        
# Demonstration of the Greedy Agent getting stuck
def run_greedy_experiment(bandit, num_steps=1000, num_runs=100): 
    """
    Run a greedy agent experiment on the bandit problem.

    :param bandit: Instance of MultiArmedBandit.
    :param num_steps: Number of steps to run the experiment.
    :param num_runs: Number of runs for averaging results.
    :return: Tuple of average regrets and standard deviation of regrets.
    """
    regrets = []
    
    for run in range(num_runs):
        agent = GreedyAgent(bandit.num_arms)
        cumulative_regret = 0
        
        # Initial exploration: try each arm once
        for arm in range(bandit.num_arms):
            reward = bandit.pull(arm)
            agent.update(arm, reward)
            regret = bandit.arm_probabilities[bandit.get_optimal_arm()] - bandit.arm_probabilities[arm]
            cumulative_regret += regret
            
        # Now run greedy
        for t in range(bandit.num_arms, num_steps):
            arm = agent.select_arm()
            reward = bandit.pull(arm)
            agent.update(arm, reward)
            regret = bandit.arm_probabilities[bandit.get_optimal_arm()] - bandit.arm_probabilities[arm]
            cumulative_regret += regret
            
        regrets.append(cumulative_regret)
        
    return np.mean(regrets), np.std(regrets)

# Ensure MultiArmedBandit is defined
if 'MultiArmedBandit' not in globals():
    class MultiArmedBandit:
        """A Simple Multi-Armed Bandit Problem Implementation."""
        def __init__(self, arm_probabilities: List[float]):
            """
            Initialize the Multi-Armed Bandit with bernoulli arms probabilities.

            :param arm_probabilities: List of probabilities for each arm.
            """
            self.arm_probabilities = arm_probabilities
            self.num_arms = len(arm_probabilities)
            self.optimal_arm = np.argmax(arm_probabilities)
        
        def pull(self, arm: int) -> int:
            """
            Pull the specified arm and return a reward (0 or 1).

            :param arm: Index of the arm to pull.
            :return: 1 if success, 0 otherwise.
            """
            if arm >= self.num_arms or arm < 0:
                raise ValueError(f"Arm index {arm} is out of bounds. Must be between 0 and {self.num_arms - 1}.")
            return int(np.random.random() < self.arm_probabilities[arm])
        
        def get_optimal_arm(self) -> int:
            """
            Get the index of the optimal arm.

            :return: Index of the optimal arm.
            """
            return np.argmax(self.arm_probabilities)

# Ensure bandit is defined
if 'bandit' not in locals():
    bandit = MultiArmedBandit([0.95, 0.90, 0.10])

mean_regret, std_regret = run_greedy_experiment(bandit)
print(f"Mean Regret: {mean_regret}, Std Dev of Regret: {std_regret}")

        
       
    
   

Mean Regret: 22.31749999999996, Std Dev of Regret: 85.70995519045748
